# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 protein abundance dataset using the `mlcroissant` library, referencing all dataset structure elements via their Croissant `@id` fields for consistency and reproducibility.

### Dataset Source
The dataset source is a Croissant schema at:
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Install mlcroissant if not available
!pip install mlcroissant

## 1. Data Loading
Load the dataset and its metadata using the `mlcroissant` library. This step fetches all metadata and enables access to record sets, fields, and files described in the Croissant schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset
dataset = mlc.Dataset(croissant_url)
# mlcroissant's metadata is attached to the .metadata attribute
metadata = dataset.metadata
print(f"Dataset title: {metadata.name}\n\n{metadata.description}\n")

## 2. Data Overview
Let's list all record sets, their field `@id`s, and provide a data structure summary (all by `@id`).

In [ ]:
# List all RecordSet `@id`s in the dataset
record_sets = list(dataset.record_sets.keys())
print("Available RecordSets (`@id`s):\n", record_sets)

# For each record set, list its fields (also by @id)
for rs_id in record_sets:
    record_set = dataset.record_sets[rs_id]
    print(f"\nRecordSet {rs_id} fields:")
    field_ids = list(record_set.fields.keys())
    for field_id in field_ids:
        field = record_set.fields[field_id]
        print(f"  - {field_id}: {field.name}")

## 3. Data Extraction
Extract records for a given RecordSet into a Pandas DataFrame, referencing by `@id`. For illustration, we extract the main protein abundance record set.

In [ ]:
# Select a record set (by @id)
# For this dataset, let's pick the first RecordSet
main_record_set_id = record_sets[0]

# Load records from this record set into a DataFrame
records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)
print(f"Fields (as DataFrame columns) for RecordSet {main_record_set_id}:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
We demonstrate EDA steps referencing DataFrame columns by their field `@id` as specified in the Croissant schema. For instance, let's filter by a numeric abundance field, normalize it, and group by a relevant annotation category if present.

In [ ]:
# Examine which fields are numeric
numeric_columns = df.select_dtypes(include=["float", "int"]).columns.tolist()
print("Numeric fields available for analysis (by @id, matches field definition):\n", numeric_columns)

# Example: Use the first numeric field for analysis
if numeric_columns:
    numeric_field_id = numeric_columns[0]
    print(f"\nAnalyzing numeric field: {numeric_field_id}")
    
    # Filter records where the numeric field > threshold
    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > mean ({threshold:.2f}): {len(filtered_df)} records")
    
    # Normalize this field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized" ]].head())
    
    # If there is a categorical/grouping field (for instance, by sample, accession, etc), pick it
    # Try for a typical 'accession' or 'sample' field
    candidate_group_fields = [col for col in df.columns if ("accession" in col.lower()) or ("sample" in col.lower())]
    group_field_id = candidate_group_fields[0] if candidate_group_fields else None
    if group_field_id:
        print(f"\nGrouping by {group_field_id} (if unique samples):")
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(grouped.head())
else:
    print("No numeric fields found in RecordSet for EDA.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field (by `@id`) and relationships with group field, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id:
        # Show boxplot for grouped data
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric columns available for visualization.")

## 6. Conclusion
We demonstrated how to use `mlcroissant` with the FAIR^2 mass spectrometry dataset, referencing all RecordSets and Fields by their Croissant `@id` for reproducibility. We loaded and explored one record set, identified numeric fields, filtered and normalized them, grouped by sample category, and visualized distributions. For deeper analysis, explore other RecordSets and relationships using their `@id`s as demonstrated above.